Eoin Devlin - 24346152

Github repo:

In [1]:
import pandas as pd
import torch
import time
import textwrap
from transformers import AutoTokenizer, AutoModelForCausalLM

In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
# try two different types of prompts, one that provides a lot of detail as an article intro and one that is quite short and generic.

PROMPT_LONG = """
Recent developments in artificial intelligence have lead to concerns about a crisis of meaning that will arise as an increasing number of jobs and creative tasks are automated away from humans. Many activities that provide meaning to people's lives such as working or creative pursuits will soon be automated.
"""

PROMPT_SHORT = """
There are many implications from recent developments in artificial intelligence.
"""

In [4]:
class Model:
    def __init__(self, name, device, prompt, **gen_kwargs):
        self.name = name
        self.device = device
        self.prompt = prompt
        self.gen_kwargs = gen_kwargs
        self.article = None
        self.time = None

    def make_model(self):
        return AutoModelForCausalLM.from_pretrained(self.name).to(self.device)

    def make_tokenizer(self):
        return AutoTokenizer.from_pretrained(self.name)

    def generate(self):
        tokenizer = self.make_tokenizer()
        model = self.make_model()

        inputs = tokenizer(
            self.prompt,
            return_tensors="pt"
        ).to(self.device)

        start = time.time()

        output = model.generate(
            **inputs,
            **self.gen_kwargs
        )

        end = time.time()

        text = tokenizer.decode(
            output[0]
        )

        duration = end-start

        self.article = text
        self.time = duration

First lets look at a basic greedy decoding using gpt2-xl model. I set max length to 128 which is just an arbitrary choice to see the first few sentences this model would generate.

In [5]:
kwargs = {
    "max_length": 128,
    "do_sample": False,
}
basic_greedy_long_prompt = Model(
    "gpt2-xl", DEVICE, PROMPT_LONG, **kwargs
)
basic_greedy_long_prompt.generate()

basic_greedy_short_prompt = Model(
    "gpt2-xl", DEVICE, PROMPT_SHORT, **kwargs
)
basic_greedy_short_prompt.generate()

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

GPT2LMHeadModel LOAD REPORT from: gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [6]:
print("\n--------------------------------------\n")
print(f"basic_greedy_short_prompt. Time: {round(basic_greedy_short_prompt.time,2)}s")
print(f"{textwrap.fill(text=basic_greedy_short_prompt.article, width=100)}")
print("\n--------------------------------------\n")
print(f"basic_greedy_long_prompt. Time: {round(basic_greedy_long_prompt.time,2)}s")
print(f"{textwrap.fill(text=basic_greedy_long_prompt.article, width=100)}")
print("\n--------------------------------------\n")


--------------------------------------

basic_greedy_short_prompt. Time: 4.47s
 There are many implications from recent developments in artificial intelligence.  The first is that
we are entering a new era of technology, one that will be defined by the ability to create and
manipulate digital information.  The second is that we are entering a new era of human-machine
interaction, one that will be defined by the ability to create and manipulate digital information.
The third is that we are entering a new era of human-machine interaction, one that will be defined
by the ability to create and manipulate digital information.  The fourth is that we are entering a
new era of human-machine interaction, one that

--------------------------------------

basic_greedy_long_prompt. Time: 3.74s
 Recent developments in artificial intelligence have lead to concerns about a crisis of meaning that
will arise as an increasing number of jobs and creative tasks are automated away from humans. Many
activi

Using a basic greedy search leads to articles with high levels of repetition with both a long and short prompt. This is expected as the greedy decoding method always selects the token with the highest conditional probability based on the preceding sequence. Setting do_sample=False leads to this deterministic token selection. This obviously would not pass as a human written article.

Now lets try a beam search using the same model and max_length and an initial num_beams=5.

In [7]:
kwargs = {
    "max_length": 128,
    "do_sample": False,
    "num_beams": 5
}
basic_beam_long_prompt = Model(
    "gpt2-xl", DEVICE, PROMPT_LONG, **kwargs
)
basic_beam_long_prompt.generate()

basic_beam_short_prompt = Model(
    "gpt2-xl", DEVICE, PROMPT_SHORT, **kwargs
)
basic_beam_short_prompt.generate()

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [8]:
print("\n--------------------------------------\n")
print(f"basic_beam_short_prompt. Time: {round(basic_beam_short_prompt.time,2)}s")
print(f"{textwrap.fill(text=basic_beam_short_prompt.article, width=100)}")
print("\n--------------------------------------\n")
print(f"basic_beam_long_prompt. Time: {round(basic_beam_long_prompt.time,2)}s")
print(f"{textwrap.fill(text=basic_beam_long_prompt.article, width=100)}")
print("\n--------------------------------------\n")


--------------------------------------

basic_beam_short_prompt. Time: 4.71s
 There are many implications from recent developments in artificial intelligence.  For example, it
is now possible to teach a computer to play the game of Go, the ancient Chinese board game.  Image
copyright Getty Images Image caption The game of Go is one of the most complex in the world  The
game of Go is one of the most complex in the world.  It is played by two players, each with a set of
black and white stones on a grid.  Each player has a set number of moves to make before the game is
over.  The first player to make a move wins.

--------------------------------------

basic_beam_long_prompt. Time: 3.71s
 Recent developments in artificial intelligence have lead to concerns about a crisis of meaning that
will arise as an increasing number of jobs and creative tasks are automated away from humans. Many
activities that provide meaning to people's lives such as working or creative pursuits will soon be
auto

A beam search leads to more sensible looking output, at least the constant repetition has stopped. However, for several reasons these are not passable as written by human.

The short prompt goes off on a tangent about about the game of Go. This is a clear indication of being written by AI because it starts aan explanation of the rules of Go which ar enot relevant to the topic in question of implications from recent AI developments. Moreover, AlphaGo first started beating professional Go players in 2015. This is not a recent development.

The long prompt spurs a more sensible article in terms of context where it discusses a study by the FHI at Oxford. However, it repeats itself and this institution continuously.



In [9]:
kwargs = {
    "max_length": 128,
    "do_sample": False,
    "num_beams": 15
}
basic_beam_2_long_prompt = Model(
    "gpt2-xl", DEVICE, PROMPT_LONG, **kwargs
)
basic_beam_2_long_prompt.generate()

basic_beam_2_short_prompt = Model(
    "gpt2-xl", DEVICE, PROMPT_SHORT, **kwargs
)
basic_beam_2_short_prompt.generate()

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [10]:
print("\n--------------------------------------\n")
print(f"basic_beam_short_prompt. Time: {round(basic_beam_2_short_prompt.time,2)}s")
print(f"{textwrap.fill(text=basic_beam_2_short_prompt.article, width=100)}")
print("\n--------------------------------------\n")
print(f"basic_beam_long_prompt. Time: {round(basic_beam_2_long_prompt.time,2)}s")
print(f"{textwrap.fill(text=basic_beam_2_long_prompt.article, width=100)}")
print("\n--------------------------------------\n")


--------------------------------------

basic_beam_short_prompt. Time: 7.11s
 There are many implications from recent developments in artificial intelligence.  Artificial
intelligence has the potential to transform the way we live, work and interact with each other.
Artificial intelligence has the potential to transform the way we live, work and interact with each
other.  Artificial intelligence has the potential to transform the way we live, work and interact
with each other.  Artificial intelligence has the potential to transform the way we live, work and
interact with each other.  Artificial intelligence has the potential to transform the way we live,
work and interact with each other.  Artificial intelligence has

--------------------------------------

basic_beam_long_prompt. Time: 5.45s
 Recent developments in artificial intelligence have lead to concerns about a crisis of meaning that
will arise as an increasing number of jobs and creative tasks are automated away from humans. 

Interestingly, increasing num_beams from 5 to 15 actually disimproves the quality of the text generation. At num_beams=5 the generated output was relatively smooth if not totally relevant to the prompt topic. Increasing num_beams=15 reverts the text generation to repetitive sentences. Intuitively this does not make sense, but it appears that by allowing a larger number of tokens to be considered at each time step the beam search reverts to re-enforcing a repetitive loop. So the overal sequence being chosen must involve tokens that lie outide of the top 5 tokens at a given time step such that this repetition is missed when num_beams=5.

Lets revert to num_beams=5 but add the constraint no_repeat_ngram_size=3 to avoid any repetition.

In [11]:
kwargs = {
    "max_length": 128,
    "do_sample": False,
    "num_beams": 5,
    "no_repeat_ngram_size":3
}
basic_beam_3_long_prompt = Model(
    "gpt2-xl", DEVICE, PROMPT_LONG, **kwargs
)
basic_beam_3_long_prompt.generate()

basic_beam_3_short_prompt = Model(
    "gpt2-xl", DEVICE, PROMPT_SHORT, **kwargs
)
basic_beam_3_short_prompt.generate()

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [12]:
print("\n--------------------------------------\n")
print(f"basic_beam_short_prompt. Time: {round(basic_beam_3_short_prompt.time,2)}s")
print(f"{textwrap.fill(text=basic_beam_3_short_prompt.article, width=100)}")
print("\n--------------------------------------\n")
print(f"basic_beam_long_prompt. Time: {round(basic_beam_3_long_prompt.time,2)}s")
print(f"{textwrap.fill(text=basic_beam_3_long_prompt.article, width=100)}")
print("\n--------------------------------------\n")


--------------------------------------

basic_beam_short_prompt. Time: 4.92s
 There are many implications from recent developments in artificial intelligence.  For example, it
is now possible to teach a computer to play the game of Go, the ancient Chinese board game that is
considered to be one of the most difficult in the world to master. In the past, it was thought that
computers would never be able to master such a complex game. However, a team of researchers from the
University of Toronto and the Chinese Academy of Sciences in Beijing have shown that it is possible
to train a Go-playing computer to beat the world's best human Go players. This is the first time
that a computer has been able to beat

--------------------------------------

basic_beam_long_prompt. Time: 3.46s
 Recent developments in artificial intelligence have lead to concerns about a crisis of meaning that
will arise as an increasing number of jobs and creative tasks are automated away from humans. Many
activities 

Adding the repetition restriction avoids the repetition of the organisation in the long prompt as seen above. This leads to a much more plausible sounding article introduction. Using studies from Universtities adds significant weight to the argument, although without citation it is not clear if this is simply made up. Regarding the short prompt, preventing repeated n-grams obviously does not prevent the model going on a tangent about Go.

Lets move to using some sampling methods rather than relying on deterministic token selection.

In [13]:
kwargs = {
    "max_length": 128,
    "do_sample": True,
    "temperature": 1.0,
    "top_p": 1.0,
    "top_k": 50
}

sample_1_long_prompt = Model(
    "gpt2-xl", DEVICE, PROMPT_LONG, **kwargs
)
sample_1_long_prompt.generate()

sample_1_short_prompt = Model(
    "gpt2-xl", DEVICE, PROMPT_SHORT, **kwargs
)
sample_1_short_prompt.generate()

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [14]:
print("\n--------------------------------------\n")
print(f"sample_1_short_prompt. Time: {round(sample_1_short_prompt.time,2)}s")
print(f"{textwrap.fill(text=sample_1_short_prompt.article, width=100)}")
print("\n--------------------------------------\n")
print(f"sample_1_long_prompt. Time: {round(sample_1_long_prompt.time,2)}s")
print(f"{textwrap.fill(text=sample_1_long_prompt.article, width=100)}")
print("\n--------------------------------------\n")


--------------------------------------

sample_1_short_prompt. Time: 4.44s
 There are many implications from recent developments in artificial intelligence.  An AI might be
able to drive an automated system to take a certain route and then find a shortcut.  It might be
able to write software that will be much more efficient than human programmers, as well as able to
understand more complex documents and understand more abstract concepts.  A machine that is a little
more human will be able to answer the myriad of questions it might have.  For example, we may be
able to have an AI monitor our activities and respond to questions and make decisions like humans
do.  Or, we might be able to

--------------------------------------

sample_1_long_prompt. Time: 2.83s
 Recent developments in artificial intelligence have lead to concerns about a crisis of meaning that
will arise as an increasing number of jobs and creative tasks are automated away from humans. Many
activities that provide meanin

Even just using the default temperature and top_p and top_k parameters it is clear that sampling methods provide a much more fluid text. There is much less repetition of phrases which is the easiest way to spot an AI-generated text. I do wonder if sampling methods can overuse rare terms, like above the use of bioelectroencephalography is pretty technical and is likely to only be found in very niche subject essays rather than a generic AI article if written by a human. Another thing that stands out to me as possibly not being human written is that the sentences tend to be quite consistent in length and cadence.

I would like to make the model a little less stiff or serious in terms of language so I will try increasing temperature and also increasing top_k to 100.

In [15]:
kwargs = {
    "max_length": 128,
    "do_sample": True,
    "temperature": 2.0,
    "top_p": 1.0,
    "top_k": 100
}

sample_2_long_prompt = Model(
    "gpt2-xl", DEVICE, PROMPT_LONG, **kwargs
)
sample_2_long_prompt.generate()

sample_2_short_prompt = Model(
    "gpt2-xl", DEVICE, PROMPT_SHORT, **kwargs
)
sample_2_short_prompt.generate()

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [16]:
print("\n--------------------------------------\n")
print(f"sample_2_short_prompt. Time: {round(sample_2_short_prompt.time,2)}s")
print(f"{textwrap.fill(text=sample_2_short_prompt.article, width=100)}")
print("\n--------------------------------------\n")
print(f"sample_2_long_prompt. Time: {round(sample_2_long_prompt.time,2)}s")
print(f"{textwrap.fill(text=sample_2_long_prompt.article, width=100)}")
print("\n--------------------------------------\n")


--------------------------------------

sample_2_short_prompt. Time: 4.37s
 There are many implications from recent developments in artificial intelligence. For people outside
a technocratic managerial class such developments raise deeply provocative and troubling
problems—for whether and HOW will those potential economic gains come about—because, as one
colleague once asked me wryly: If robots leave people alone, will working families get one of those
sweet new bonuses during their retirement... now with a 1 percent overhead . Meanwhile our entire
scientific basis and human identity could be severely hobbled; one must already ask whether, over
time, not creating a "singular robot" in isolation . Here there could be social implications--how
the job of AI does

--------------------------------------

sample_2_long_prompt. Time: 2.81s
 Recent developments in artificial intelligence have lead to concerns about a crisis of meaning that
will arise as an increasing number of jobs and creati

The generated text is quite clearly not written by a human. It is jarring and incoherent. Clearly I have allowed too much flexibility in the model. Lets try reducing temperature again but keeping it above 1.0 so the model is still "creative". However, I will reduce top_k and top_p so that only more meaningful tokens are available for selection.

In [17]:
kwargs = {
    "max_length": 128,
    "do_sample": True,
    "temperature": 1.25,
    "top_p": 0.7,
    "top_k": 25
}

sample_3_long_prompt = Model(
    "gpt2-xl", DEVICE, PROMPT_LONG, **kwargs
)
sample_3_long_prompt.generate()

sample_3_short_prompt = Model(
    "gpt2-xl", DEVICE, PROMPT_SHORT, **kwargs
)
sample_3_short_prompt.generate()

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [18]:
print("\n--------------------------------------\n")
print(f"sample_3_short_prompt. Time: {round(sample_3_short_prompt.time,2)}s")
print(f"{textwrap.fill(text=sample_3_short_prompt.article, width=100)}")
print("\n--------------------------------------\n")
print(f"sample_3_long_prompt. Time: {round(sample_3_long_prompt.time,2)}s")
print(f"{textwrap.fill(text=sample_3_long_prompt.article, width=100)}")
print("\n--------------------------------------\n")


--------------------------------------

sample_3_short_prompt. Time: 4.36s
 There are many implications from recent developments in artificial intelligence.  In the first
place, it may mean that, for example, we may be able to program computers not just to be able to
perform tasks but to think, and to do so in the right way. It may mean that the computers we have
now will be able to do more, be better and do things better than any human.  In the second place, AI
could be the key to unlocking the secrets of the universe. If machines could think and reason in a
similar way to humans, then they could potentially understand the secrets of the universe in the
same way

--------------------------------------

sample_3_long_prompt. Time: 2.85s
 Recent developments in artificial intelligence have lead to concerns about a crisis of meaning that
will arise as an increasing number of jobs and creative tasks are automated away from humans. Many
activities that provide meaning to people's lives su

This iteration has more coherence in the generated text, but certain patterns stand out as AI generated. Firstly, repeatedly starting a sentence with "This is because". And secondly every sentence follows the same structure of beginning with a conjunction and then explaining a reasoning. It does not read like a fluid piece that connects the same thought across multiple sentences.

In [19]:
kwargs = {
    "max_length": 128,
    "do_sample": True,
    "temperature": 1.25,
    "top_p": 0.7,
    "top_k": 75
}

sample_4_long_prompt = Model(
    "gpt2-xl", DEVICE, PROMPT_LONG, **kwargs
)
sample_4_long_prompt.generate()

sample_4_short_prompt = Model(
    "gpt2-xl", DEVICE, PROMPT_SHORT, **kwargs
)
sample_4_short_prompt.generate()

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [20]:
print("\n--------------------------------------\n")
print(f"sample_4_short_prompt. Time: {round(sample_4_short_prompt.time,2)}s")
print(f"{textwrap.fill(text=sample_4_short_prompt.article, width=100)}")
print("\n--------------------------------------\n")
print(f"sample_4_long_prompt. Time: {round(sample_4_long_prompt.time,2)}s")
print(f"{textwrap.fill(text=sample_4_long_prompt.article, width=100)}")
print("\n--------------------------------------\n")


--------------------------------------

sample_4_short_prompt. Time: 4.49s
 There are many implications from recent developments in artificial intelligence.  But one of the
most important is that machine learning will create a huge market for AI tools, both big and small.
Image caption The market for computer vision is already worth billions of dollars  "Machine learning
will not replace people," said Mr Pashley.  "It will complement and complement humans."  But the
most immediate change is the rapid advance in machine learning, with applications in everything from
search and security, to driving, manufacturing and financial services.  In the past, the industry
focused on solving hard problems, he said.

--------------------------------------

sample_4_long_prompt. Time: 2.89s
 Recent developments in artificial intelligence have lead to concerns about a crisis of meaning that
will arise as an increasing number of jobs and creative tasks are automated away from humans. Many
activities 

By increasing top_k to 75 some fluidity is returned to the article. Both the short and long prompt articles read as if a shared thought ties sentences together. There is also less repetition of phrases and words and real life examples are used in both cases.

Lets try reducing temperature to below 1 which should make the model select higher probability words more often. I will only consider the top 50 at each step and set top_p=0.9 to avoid any words in the long right tail.

Finally, lets also add a repetition penalty of 1.1 to reduce any repetition.

In [21]:
kwargs = {
    "max_length": 128,
    "do_sample": True,
    "temperature": 0.7,
    "top_p": 0.9,
    "top_k": 50,
    "repetition_penalty": 1.1
}

sample_5_long_prompt = Model(
    "gpt2-xl", DEVICE, PROMPT_LONG, **kwargs
)
sample_5_long_prompt.generate()

sample_5_short_prompt = Model(
    "gpt2-xl", DEVICE, PROMPT_SHORT, **kwargs
)
sample_5_short_prompt.generate()

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [22]:
print("\n--------------------------------------\n")
print(f"sample_5_short_prompt. Time: {round(sample_5_short_prompt.time,2)}s")
print(f"{textwrap.fill(text=sample_5_short_prompt.article, width=100)}")
print("\n--------------------------------------\n")
print(f"sample_5_long_prompt. Time: {round(sample_5_long_prompt.time,2)}s")
print(f"{textwrap.fill(text=sample_5_long_prompt.article, width=100)}")
print("\n--------------------------------------\n")


--------------------------------------

sample_5_short_prompt. Time: 4.46s
 There are many implications from recent developments in artificial intelligence.  We have already
seen that the use of AI can increase productivity and increase output – but at what cost? It is not
clear what a fully automated future will look like, or how this might be controlled.  The potential
for increased automation and its impact on society has been well-recognised by economists, and it is
an issue that has been raised during the Brexit campaign.  In one of the most famous studies,
economist Thomas Piketty argued that the wealth gap between the richest 1 per cent of households and
the rest was driven largely by increasing inequality in

--------------------------------------

sample_5_long_prompt. Time: 2.85s
 Recent developments in artificial intelligence have lead to concerns about a crisis of meaning that
will arise as an increasing number of jobs and creative tasks are automated away from humans. Man

This is by far the best paramet set. There is little obvious repetition in the generated articles. The generated text stays on topic to the prompt. There are concrete examples provided for points made in the text, and differing sentence structures are used such as rhetorical questions.

So far I have only used the gpt2-xl model. Lets try some other ones. I picked models with a parameter set < 1B so I can load and test the experiments quickly. Loading large models is time consuming and did not feel necessary for this exercise.

In [29]:
kwargs = {
    "max_length": 128,
    "do_sample": True,
    "temperature": 0.7,
    "top_p": 0.9,
    "top_k": 50,
    "repetition_penalty": 1.1
}
liquidai_long_prompt = Model(
    "LiquidAI/LFM2.5-350M", DEVICE, PROMPT_LONG, **kwargs
)
liquidai_long_prompt.generate()

liquidai_short_prompt = Model(
    "LiquidAI/LFM2.5-350M", DEVICE, PROMPT_SHORT, **kwargs
)
liquidai_short_prompt.generate()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [30]:
print("\n--------------------------------------\n")
print(f"liquidai_short_prompt. Time: {round(liquidai_short_prompt.time,2)}s")
print(f"{textwrap.fill(text=liquidai_short_prompt.article, width=100)}")
print("\n--------------------------------------\n")
print(f"liquidai_long_prompt. Time: {round(liquidai_long_prompt.time,2)}s")
print(f"{textwrap.fill(text=liquidai_long_prompt.article, width=100)}")
print("\n--------------------------------------\n")


--------------------------------------

liquidai_short_prompt. Time: 2.5s
<|startoftext|> There are many implications from recent developments in artificial intelligence. The
first thing to note is that the initial introduction of AI was introduced by a team of researchers
at the University of Michigan and the University of Pennsylvania, led by Dr. Richard H. Morrissey,
who also served as the head of the research group.  The second thing to note is that the initial
introduction of AI was introduced by a team of researchers at the University of Michigan and the
University of Pennsylvania, led by Dr. Richard H. Morrissey, who also served as the head of the
research group.  The third thing to note is that the second introduction of

--------------------------------------

liquidai_long_prompt. Time: 1.62s
<|startoftext|> Recent developments in artificial intelligence have lead to concerns about a crisis
of meaning that will arise as an increasing number of jobs and creative tasks are aut

LiquidAI/LFM2.5-350M performs reasonably well on the long prompt. The response to the short prompt is not relevant to AI implications and ahas a high level of repeititon despite the penalty.

In [31]:
kwargs = {
    "max_length": 128,
    "do_sample": True,
    "temperature": 0.7,
    "top_p": 0.9,
    "top_k": 50,
    "repetition_penalty": 1.1
}
tiny_llama_long_prompt = Model(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0", DEVICE, PROMPT_LONG, **kwargs
)
tiny_llama_long_prompt.generate()

tiny_llama_short_prompt = Model(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0", DEVICE, PROMPT_SHORT, **kwargs
)
tiny_llama_short_prompt.generate()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [32]:
print("\n--------------------------------------\n")
print(f"tiny_llama_short_prompt. Time: {round(tiny_llama_short_prompt.time,2)}s")
print(f"{textwrap.fill(text=tiny_llama_short_prompt.article, width=100)}")
print("\n--------------------------------------\n")
print(f"tiny_llama_long_prompt. Time: {round(tiny_llama_long_prompt.time,2)}s")
print(f"{textwrap.fill(text=tiny_llama_long_prompt.article, width=100)}")
print("\n--------------------------------------\n")


--------------------------------------

tiny_llama_short_prompt. Time: 3.4s
<s>  There are many implications from recent developments in artificial intelligence.  1. Job
displacement: The rise of automation is expected to lead to the loss of millions of jobs worldwide.
This could lead to a shift in the global labor market, with some countries becoming more competitive
and others struggling to adapt.  2. Economic growth: Automation could help boost economic growth by
improving productivity and reducing costs. However, it may also result in wage stagnation, as
workers are displaced by new technologies.  3. Social inequality: As automation increases,

--------------------------------------

tiny_llama_long_prompt. Time: 2.15s
<s>  Recent developments in artificial intelligence have lead to concerns about a crisis of meaning
that will arise as an increasing number of jobs and creative tasks are automated away from humans.
Many activities that provide meaning to people's lives such as work

TinyLlama/TinyLlama-1.1B-Chat-v1.0 also performs well. The response to the short prompt simply lists relevant implications which does not read as an article. The response to the longer prompt reads well with quotations from sources and little repetition. It also runs 25% faster than gpt2-xl.

The optimal model and parameter set for this task is one that balances coherence (lower temperature, lower top_p and top_k) with some element of randomness (higher temp and repetition penalty). While I don't have any metrics for measuring how "good" AI generated text is, I was looking out for repetition of words and phrases, use of niche words that humans rarely use, and repetitive sentence structure. I settled on a gpt2-xl model with these parameters

In [32]:
kwargs = {
    "max_length": 128,
    "do_sample": True,
    "temperature": 0.7,
    "top_p": 0.9,
    "top_k": 50,
    "repetition_penalty": 1.1
}